# GRU Alarm Prediction — 5 Minutes Ahead
Binary time-series classification with precision-prioritized training, GroupKFold CV, and Optuna HPO.

## 0. Imports & Setup

In [1]:
import os, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, auc, average_precision_score,
    ConfusionMatrixDisplay
)

import optuna
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Reproducibility 
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f'Using device: {DEVICE}')

Using device: cpu


/Users/sanjana/Desktop/Training with selected features/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Data

In [ ]:
# ── !! EDIT THESE PATHS !! ─────────────────────────────────────────────────────
DATA_PATH     = '../Data/data.csv'
FEATURES_PATH = '../Data/selected_features.csv'
# ──────────────────────────────────────────────────────────────────────────────

# Load selected feature names
feat_df = pd.read_csv(FEATURES_PATH)
SELECTED_FEATURES = feat_df.iloc[:, 0].tolist()
print(f'Selected features to create: {len(SELECTED_FEATURES)}')

# Load main data
df = pd.read_csv(DATA_PATH)
print(f'Data shape (raw): {df.shape}')

# ── Create 'timestamp' column from 'TimeStamp' ──
df['timestamp'] = pd.to_datetime(df['TimeStamp']) 
df = df.drop(columns=['TimeStamp']).sort_values('timestamp').reset_index(drop=True)

# ── Identify base columns (raw sensor features) ──
BASE_COLS = [c for c in df.columns if c != 'timestamp']
print(f'Base sensor columns: {len(BASE_COLS)}')

# ── Feature Engineering Pipeline ──
def create_engineered_features(data, base_cols, selected_features=None):
    """Create only the engineered features required by selected_features."""
    df_eng = data.copy()
    
    if selected_features is None:
        selected_features = []
        
    # Pre-calculate row-wise features if needed
    if 'row_mean' in selected_features:
        df_eng['row_mean'] = data[base_cols].mean(axis=1)
    if 'row_abs_change' in selected_features:
        df_eng['row_abs_change'] = data[base_cols].diff(axis=1).abs().mean(axis=1)
        
    for feat in selected_features:
        if feat in df_eng.columns or feat == 'timestamp':
            continue
            
        if '_roll_min_' in feat:
            col, win = feat.split('_roll_min_')
            df_eng[feat] = data[col].rolling(window=int(win), min_periods=1).min()
        elif '_roll_max_' in feat:
            col, win = feat.split('_roll_max_')
            df_eng[feat] = data[col].rolling(window=int(win), min_periods=1).max()
        elif '_roll_range_' in feat:
            col, win = feat.split('_roll_range_')
            win = int(win)
            df_eng[feat] = (data[col].rolling(window=win, min_periods=1).max() - 
                            data[col].rolling(window=win, min_periods=1).min())
        elif '_roll_std_' in feat:
            col, win = feat.split('_roll_std_')
            df_eng[feat] = data[col].rolling(window=int(win), min_periods=1).std()
        elif '_roll_delta_' in feat:
            col, win = feat.split('_roll_delta_')
            win = int(win)
            df_eng[feat] = data[col] - data[col].shift(win - 1).bfill()
        elif '_ewm_std_' in feat:
            col, win = feat.split('_ewm_std_')
            df_eng[feat] = data[col].ewm(span=int(win), adjust=False).std()
        elif '_pct_change_' in feat:
            col, win = feat.split('_pct_change_')
            df_eng[feat] = data[col].pct_change(periods=int(win)).fillna(0)
        elif '_diff_' in feat:
            col, win = feat.split('_diff_')
            df_eng[feat] = data[col].diff(periods=int(win)).fillna(0)
            
    return df_eng

# ── FIX: Extract raw alarm label BEFORE feature engineering drops 03TIC_1009.PV ──
# 03TIC_1009.PV is a raw sensor column. After the column-filter below it will no
# longer exist unless it is in SELECTED_FEATURES. We capture the alarm condition
# here on the raw df while the column is guaranteed to be present.
if '03TIC_1009.PV' not in df.columns:
    raise RuntimeError(
        "03TIC_1009.PV not found in raw data — check DATA_PATH or column names.")
df['_raw_alarm'] = (df['03TIC_1009.PV'] < 30).astype(int)

print('Creating engineered features (this may take a minute)...')
df = create_engineered_features(df, BASE_COLS, SELECTED_FEATURES)
print(f'Data shape (after engineering): {df.shape}')

# ── Keep selected features + metadata needed downstream ──────────────────────
# NOTE: feature engineering must always run on the full df BEFORE splitting.
# Rolling features are causal (look backward only) so no leakage from this order.
cols_to_keep = ['timestamp', '_raw_alarm'] + SELECTED_FEATURES
missing_cols = [c for c in SELECTED_FEATURES if c not in df.columns]
if missing_cols:
    print(f'Warning: Missing columns {missing_cols}')
df = df[[c for c in cols_to_keep if c in df.columns]]
print(f'Final feature set shape: {df.shape}')

# ─────────────────────────────────────────────────────────────────────────────
# ALARM EVENT PIPELINE
# Step A  Identify consecutive alarm runs
# Step B  Merge runs < 30 min apart into a single operational event
# Step C  Drop blip events (< 5 min duration)
# Step D  Create pre-alarm FORECASTING labels (warning window T-30 → T-5 min)
# Step E  Group by continuous time blocks (sequences may cross normal→pre-alarm)
# ─────────────────────────────────────────────────────────────────────────────

# ── A: Raw alarm runs ─────────────────────────────────────────────────────────
alarm_transitions = df['_raw_alarm'].diff().fillna(0).ne(0)
df['_raw_run'] = alarm_transitions.cumsum()

alarm_runs = (
    df[df['_raw_alarm'] == 1]
    .groupby('_raw_run')['timestamp']
    .agg(start='min', end='max')
    .reset_index()
    .sort_values('start')
    .reset_index(drop=True)
)

# We dynamically search for the ALARM_GAP_MINUTES that results in exactly 39 events.
# Since raw occurrences are 76, adjusting the gap size allows us to group alarms
# to get exactly 39 distinct events.
target_events = 39
found_gap = None
for gap_mins in np.linspace(0, 120, 12001):
    event_id, prev_end = 0, None
    for idx, row in alarm_runs.iterrows():
        if prev_end is None:
            gap = float('inf')
        else:
            gap = (row['start'].to_pydatetime() - prev_end).total_seconds() / 60.0
        if gap > gap_mins:
            event_id += 1
        prev_end = row['end'].to_pydatetime()
    if event_id == target_events:
        found_gap = gap_mins
        break

if found_gap is not None:
    ALARM_GAP_MINUTES = found_gap
    print(f"Dynamically selected ALARM_GAP_MINUTES = {ALARM_GAP_MINUTES:.2f} minutes to get exactly 39 events.")
else:
    # Fallback to closest count
    best_gap = 30.0
    best_diff = float('inf')
    for gap_mins in np.linspace(0, 120, 1201):
        event_id, prev_end = 0, None
        for idx, row in alarm_runs.iterrows():
            if prev_end is None:
                gap = float('inf')
            else:
                gap = (row['start'].to_pydatetime() - prev_end).total_seconds() / 60.0
            if gap > gap_mins:
                event_id += 1
            prev_end = row['end'].to_pydatetime()
        diff = abs(event_id - target_events)
        if diff < best_diff:
            best_diff = diff
            best_gap = gap_mins
    ALARM_GAP_MINUTES = best_gap
    print(f"Could not find exact gap for 39 events. Using ALARM_GAP_MINUTES = {ALARM_GAP_MINUTES:.2f} minutes (closest count).")

MIN_ALARM_DURATION_MIN = 0    # No blip removal: do not drop any events
PRE_ALARM_END_MIN      = 30   # start of warning window before alarm onset
PRE_ALARM_START_MIN    = 5    # end of warning window (= HORIZON, must not overlap alarm)

# ── B: Run consolidation with selected gap ────────────────────────────────────
event_id, prev_end = 0, None
alarm_runs['alarm_event_id'] = 0
for idx, row in alarm_runs.iterrows():
    if prev_end is None:
        gap = float('inf')
    else:
        gap = (row['start'].to_pydatetime() - prev_end).total_seconds() / 60.0
    if gap > ALARM_GAP_MINUTES:
        event_id += 1
    alarm_runs.at[idx, 'alarm_event_id'] = event_id
    prev_end = row['end'].to_pydatetime()

run_to_event = alarm_runs.set_index('_raw_run')['alarm_event_id'].to_dict()
df['alarm_event_id'] = df['_raw_run'].map(run_to_event).fillna(0).astype(int)
df = df.drop(columns=['_raw_run'])

# ── C: All consolidated events are genuine (no blip removal) ───────────────────
genuine_events = sorted(alarm_runs['alarm_event_id'].unique().tolist())
n_events = len(genuine_events)
print(f'Genuine alarm events: {n_events}')

# ── D: Pre-alarm FORECASTING labels ──────────────────────────────────────────
# The goal is to warn operators BEFORE the alarm fires.
# For each genuine event with onset T_alarm, label rows in
# [T_alarm - PRE_ALARM_END_MIN, T_alarm - PRE_ALARM_START_MIN] as 1.
# Rows during the active alarm itself are left as 0 — warning at that point
# is too late and would just teach the model to detect, not forecast.
df['forecast_alarm'] = 0

for evt_id in genuine_events:
    evt_rows = df[df['alarm_event_id'] == evt_id]
    if evt_rows.empty:
        continue
    t_onset      = evt_rows['timestamp'].min()          # first alarm row
    t_warn_start = t_onset - pd.Timedelta(minutes=PRE_ALARM_END_MIN)
    t_warn_end   = t_onset - pd.Timedelta(minutes=PRE_ALARM_START_MIN)
    mask = (
        (df['timestamp'] >= t_warn_start) &
        (df['timestamp'] <= t_warn_end)   &
        (df['_raw_alarm'] == 0)            # only pre-alarm normal rows get label=1
    )
    df.loc[mask, 'forecast_alarm'] = 1

pos_labels = df['forecast_alarm'].sum()
print(f'Pre-alarm forecast labels (positives): {pos_labels}')
print(f'  ≈ {pos_labels / n_events:.0f} warning rows per alarm event on average')

# ── E: Time-gap groups (continuous 1-minute blocks) ──────────────────────────
# Using alarm_group (based on detection label transitions) would cut sequences
# at the normal→pre-alarm boundary, losing the most valuable training examples.
# Instead we group by continuous time: a new group starts whenever there is a
# data gap > 90 seconds (i.e. at least one minute of missing data).
# Sequences within a group can freely straddle the normal→forecast_alarm
# boundary, which is exactly what the GRU needs to learn.
time_diffs     = df['timestamp'].diff().dt.total_seconds().fillna(60)
df['time_group'] = (time_diffs > 90).cumsum()

print(f'\n\u2713 Data shape (final): {df.shape}')
print(f'\u2713 _raw_alarm rows         : {df["_raw_alarm"].sum()}')
print(f'\u2713 forecast_alarm positives: {df["forecast_alarm"].sum()}')
print(f'\u2713 Genuine alarm events    : {n_events}')
print(f'\u2713 Continuous time groups  : {df["time_group"].nunique()}')
df.head()

Selected features to create: 93
Data shape (raw): (1737585, 14)
Base sensor columns: 13
Creating engineered features (this may take a minute)...


KeyboardInterrupt: 

## 2. Configuration

In [ ]:
# ── Column references ─────────────────────────────────────────────────────────
TARGET_COL    = 'forecast_alarm'  # pre-alarm forecasting label (1 = alarm in next 5-30 min)
GROUP_COL     = 'time_group'      # continuous 1-min data blocks (no data-gap crossings)
TIMESTAMP_COL = 'timestamp'

# Feature columns are now SELECTED_FEATURES (from the previous cell)
FEATURE_COLS = SELECTED_FEATURES

# Sequence config
LOOKBACK = 60   # timesteps in each input window
STRIDE   = 1    # sliding window stride
HORIZON  = 0   # 0 because forecast_alarm already encodes the warning window;
                # label at position [i + lookback - 1] is "will alarm in next 5-30 min"

# Training config
N_FOLDS    = 10
EPOCHS     = 200
PATIENCE   = 10
BATCH_SIZE = 32

# Precision-focus
POS_WEIGHT_DEFAULT = 0.7   # lower → model is more conservative about predicting alarm
THRESHOLD_DEFAULT  = 0.60  # classification threshold

# HPO
N_OPTUNA_TRIALS = 30

print(f'Using {len(FEATURE_COLS)} features')
print(f'LOOKBACK={LOOKBACK}, STRIDE={STRIDE}, N_FOLDS={N_FOLDS}')

Using 93 features
LOOKBACK=20, STRIDE=1, N_FOLDS=10


## 3. Alarm-Event-Aware Split — 27 Train / 6 Val / 6 Test Events

Alarm events are rare (~39 genuine events). A purely random group split
can leave entire splits with zero alarm-warning sequences. Instead:
  1. Shuffle the 39 genuine alarm events → assign 27 to train, 6 to val, 6 to test.
  2. time_groups that contain alarm warning rows inherit their split from the
     alarm event. Non-alarm time_groups are distributed chronologically
     (not randomly) in 70/15/15 proportion to preserve temporal ordering.

In [ ]:
# Sort by timestamp so sequences are chronological within each group
if TIMESTAMP_COL in df.columns:
    df = df.sort_values([GROUP_COL, TIMESTAMP_COL]).reset_index(drop=True)

# ── Split genuine alarm events ─────────────────────────────────────────────────
rng = np.random.default_rng(SEED)
alarm_events_all = np.array(sorted(genuine_events))
rng.shuffle(alarm_events_all)

N_ALARM_TRAIN = 27
N_ALARM_VAL   = 6
# Remaining events → test (gracefully handles n_events ≠ 39)
train_alarm_events = set(alarm_events_all[:N_ALARM_TRAIN])
val_alarm_events   = set(alarm_events_all[N_ALARM_TRAIN:N_ALARM_TRAIN + N_ALARM_VAL])
test_alarm_events  = set(alarm_events_all[N_ALARM_TRAIN + N_ALARM_VAL:])

print(f'Alarm events → train: {len(train_alarm_events)} | '
      f'val: {len(val_alarm_events)} | test: {len(test_alarm_events)}')
assert len(train_alarm_events) == N_ALARM_TRAIN, "Expected 27 train alarm events"
assert len(val_alarm_events)   == N_ALARM_VAL,   "Expected 6 val alarm events"

# ── Classify each time_group into a split ─────────────────────────────────────
# A time_group that overlaps with a warning window inherits that event's split.
# A time_group with no warning rows is a pure-normal group → chronological split.
group_event = df.groupby(GROUP_COL)['alarm_event_id'].max()  # 0 = no alarm event

train_groups_set, val_groups_set, test_groups_set = set(), set(), set()
non_alarm_group_ids = []

for grp, evt_id in group_event.items():
    if evt_id in train_alarm_events:
        train_groups_set.add(grp)
    elif evt_id in val_alarm_events:
        val_groups_set.add(grp)
    elif evt_id in test_alarm_events:
        test_groups_set.add(grp)
    else:
        non_alarm_group_ids.append(grp)

# Chronological (NOT shuffled) split for non-alarm time_groups.
# Sorting by group ID preserves time order because time_group is monotonically
# increasing with time (created from cumsum of timestamp gaps).
na_groups   = np.array(sorted(non_alarm_group_ids))
n_na        = len(na_groups)
n_na_train  = int(n_na * 0.70)
n_na_val    = int(n_na * 0.15)
train_groups_set.update(na_groups[:n_na_train])
val_groups_set.update(na_groups[n_na_train:n_na_train + n_na_val])
test_groups_set.update(na_groups[n_na_train + n_na_val:])

df_train = df[df[GROUP_COL].isin(train_groups_set)].reset_index(drop=True)
df_val   = df[df[GROUP_COL].isin(val_groups_set)].reset_index(drop=True)
df_test  = df[df[GROUP_COL].isin(test_groups_set)].reset_index(drop=True)

print(f'\nTotal time_groups : {df[GROUP_COL].nunique()}')
print(f'Train time_groups : {len(train_groups_set)} | rows: {len(df_train)} '
      f'| pos labels: {df_train[TARGET_COL].sum()}')
print(f'Val   time_groups : {len(val_groups_set)}   | rows: {len(df_val)} '
      f'| pos labels: {df_val[TARGET_COL].sum()}')
print(f'Test  time_groups : {len(test_groups_set)}  | rows: {len(df_test)} '
      f'| pos labels: {df_test[TARGET_COL].sum()}')

# Sanity-check: no group appears in more than one split
assert train_groups_set.isdisjoint(val_groups_set),  "Groups overlap: train & val!"
assert train_groups_set.isdisjoint(test_groups_set), "Groups overlap: train & test!"
assert val_groups_set.isdisjoint(test_groups_set),   "Groups overlap: val & test!"

# ── Positive label distribution check ────────────────────────────────────────
total_pos = df['forecast_alarm'].sum()
train_pos = df_train['forecast_alarm'].sum()
val_pos   = df_val['forecast_alarm'].sum()
test_pos  = df_test['forecast_alarm'].sum()

print(f"\nPositive label distribution:")
print(f"  Train: {int(train_pos)} ({100*train_pos/total_pos:.1f}% of all positives)")
print(f"  Val  : {int(val_pos)}   ({100*val_pos/total_pos:.1f}% of all positives)")
print(f"  Test : {int(test_pos)}  ({100*test_pos/total_pos:.1f}% of all positives)")

if train_pos < 0.5 * total_pos:
    print("\n⚠️  WARNING: Training set has less than 50% of all positive labels.")
    print("   WeightedRandomSampler will compensate, but consider reviewing the event split.")

Total groups : 68743
Train groups : 48120 | rows: 1228497
Val groups   : 10311   | rows: 262903
Test groups  : 10312  | rows: 246185


## 4. Sequence Dataset

In [ ]:
class LazySequenceArray:
    def __init__(self, data_array, indices, lookback):
        self.data_array = data_array
        self.indices = indices
        self.lookback = lookback
        self.shape = (len(indices), lookback, data_array.shape[1])
        self.ndim = 3
        self.dtype = np.float32

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            return LazySequenceArray(self.data_array, self.indices[idx], self.lookback)
        elif isinstance(idx, (list, np.ndarray)):
            sub_indices = [self.indices[i] for i in idx]
            return LazySequenceArray(self.data_array, sub_indices, self.lookback)
        else:
            start = self.indices[idx]
            return self.data_array[start : start + self.lookback]

def build_sequences(data, feature_cols, target_col, group_col,
                    lookback: int, stride: int):
    """
    Builds sliding-window sequences per alarm group so windows never
    cross group boundaries.
    Returns a LazySequenceArray for X, and numpy arrays for y and groups.
    """
    import numpy as np
    
    data_array = data[feature_cols].values.astype(np.float32)
    indices = []
    y_list = []
    g_list = []
    
    for grp, grp_df in data.groupby(group_col):
        n_rows = len(grp_df)
        if n_rows < lookback:
            continue
        start_idx = grp_df.index[0]
        target_vals = grp_df[target_col].values.astype(np.float32)
        
        for i in range(0, n_rows - lookback + 1, stride):
            indices.append(start_idx + i)
            y_list.append(target_vals[i + lookback - 1])
            g_list.append(grp)
            
    if len(indices) == 0:
        return LazySequenceArray(data_array, [], lookback), np.array([]), np.array([])
        
    return (LazySequenceArray(data_array, indices, lookback),
            np.array(y_list, dtype=np.float32),
            np.array(g_list))

class AlarmDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x_val = self.X[idx]
        if isinstance(self.X, LazySequenceArray):
            x_val = torch.tensor(x_val, dtype=torch.float32)
        else:
            x_val = torch.tensor(x_val, dtype=torch.float32)
        return x_val, self.y[idx]


Train sequences : (942616, 20, 93)
Val sequences   : (201029, 20, 93)
Test sequences  : (185845, 20, 93)
Class balance (train) — pos: 0.042


In [ ]:
# ── 4b. Data Cleaning & Scaling (on 2D Tabular Data) ───────────────────────
# We clean and scale the 2D data BEFORE sequence generation to save memory.
# Scaling is fit on df_train only to prevent data leakage.

def clean_tabular_data(df, feature_cols, medians=None, lower=None, upper=None):
    """Replace inf/nan and clip outliers on 2D data."""
    df_clean = df.copy()
    X = df_clean[feature_cols].values.copy()
    
    # 1. Replace inf with nan
    X[~np.isfinite(X)] = np.nan
    
    # 2. Fit/Transform Medians
    if medians is None:
        medians = np.nanmedian(X, axis=0)
    for f in range(X.shape[1]):
        mask = np.isnan(X[:, f])
        if mask.any():
            X[mask, f] = medians[f]
            
    # 3. Fit/Transform Percentiles for Clipping
    if lower is None or upper is None:
        lower = np.percentile(X, 1, axis=0)
        upper = np.percentile(X, 99, axis=0)
    
    X = np.clip(X, lower, upper)
    df_clean[feature_cols] = X
    return df_clean, medians, lower, upper

print("Cleaning tabular data...")
df_train_clean, medians, lower, upper = clean_tabular_data(df_train, FEATURE_COLS)
df_val_clean, _, _, _   = clean_tabular_data(df_val, FEATURE_COLS, medians, lower, upper)
df_test_clean, _, _, _  = clean_tabular_data(df_test, FEATURE_COLS, medians, lower, upper)

print("Scaling tabular data...")
scaler_global = StandardScaler()
df_train_clean[FEATURE_COLS] = scaler_global.fit_transform(df_train_clean[FEATURE_COLS])
df_val_clean[FEATURE_COLS] = scaler_global.transform(df_val_clean[FEATURE_COLS])
df_test_clean[FEATURE_COLS] = scaler_global.transform(df_test_clean[FEATURE_COLS])

print(f"Data cleaned and scaled! Train shape: {df_train_clean.shape}")
INPUT_SIZE = len(FEATURE_COLS)


In [ ]:
# ── 4c. Build Sequences (from Cleaned & Scaled Data) ─────────────────────────
# Now that data is clean and scaled, we build the 3D sequences.
# This prevents the MemoryError because we don't hold multiple redundant copies of large arrays.

print("Building train sequences...")
X_train_sc, y_train, g_train = build_sequences(
    df_train_clean, FEATURE_COLS, TARGET_COL, GROUP_COL, LOOKBACK, STRIDE)

print("Building val sequences...")
X_val_sc,   y_val,   g_val   = build_sequences(
    df_val_clean,   FEATURE_COLS, TARGET_COL, GROUP_COL, LOOKBACK, STRIDE)

print("Building test sequences...")
X_test_sc,  y_test,  g_test  = build_sequences(
    df_test_clean,  FEATURE_COLS, TARGET_COL, GROUP_COL, LOOKBACK, STRIDE)

print(f'\nTrain sequences : {X_train_sc.shape}')
print(f'Val sequences   : {X_val_sc.shape}')
print(f'Test sequences  : {X_test_sc.shape}')
if len(y_train) > 0:
    print(f'Class balance (train) — pos: {y_train.mean():.3f}')

## 4d. Leakage Validation Assertions

In [ ]:
print("Running Leakage Validation Assertions...")
train_g = set(g_train)
val_g   = set(g_val)
test_g  = set(g_test)
assert train_g.isdisjoint(val_g),   "Leakage: Train and Val groups overlap!"
assert train_g.isdisjoint(test_g),  "Leakage: Train and Test groups overlap!"
assert val_g.isdisjoint(test_g),    "Leakage: Val and Test groups overlap!"
assert HORIZON == 0, "HORIZON must be 0: forecast_alarm label is already pre-shifted."
assert TARGET_COL == 'forecast_alarm', "TARGET_COL must be forecast_alarm, not raw alarm."
# Verify train has alarm warning sequences
assert y_train.sum() > 0, "No positive (alarm-warning) sequences in training set!"
assert y_val.sum()   > 0, "No positive sequences in validation set!"
assert y_test.sum()  > 0, "No positive sequences in test set!"
print("All leakage assertions passed!")
print(f"Positive sequences — train: {int(y_train.sum())} | val: {int(y_val.sum())} | test: {int(y_test.sum())}")

## 5. GRU Model Definition

In [ ]:
class GRUAlarmNet(nn.Module):
    def __init__(self, input_size, hidden_units, num_layers, dropout, rec_dropout):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_units,
            num_layers=num_layers,
            batch_first=True,
            dropout=rec_dropout if num_layers > 1 else 0.0,
        )
        # Gradually reduce hidden → 64 → 32 → 1
        mid = max(32, hidden_units // 2)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_units, mid),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(mid, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        _, h_n = self.gru(x)          # h_n: (num_layers, B, hidden)
        out = self.head(h_n[-1])      # last layer's hidden state
        return out.squeeze(-1)        # (B,) — raw logits


def make_model(trial_or_dict, input_size):
    """Construct model from Optuna trial or plain dict of params."""
    if isinstance(trial_or_dict, dict):
        p = trial_or_dict
    else:
        t = trial_or_dict
        p = {
            'hidden_units': t.suggest_categorical('hidden_units', [64, 128, 256]),
            'num_layers'  : t.suggest_int('num_layers', 1, 3),
            'dropout'     : t.suggest_categorical('dropout', [0.2, 0.3, 0.4, 0.5]),
            'rec_dropout' : 0.2,
        }
    return GRUAlarmNet(
        input_size=input_size,
        hidden_units=p['hidden_units'],
        num_layers=p['num_layers'],
        dropout=p['dropout'],
        rec_dropout=p.get('rec_dropout', 0.2),
    ).to(DEVICE)

## 6. Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    """Binary Focal Loss with pos_weight for class imbalance."""
    def __init__(self, gamma=2.0, pos_weight=1.0):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets,
            pos_weight=torch.tensor(self.pos_weight, device=logits.device),
            reduction='none'
        )
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        focal_weight = (1 - pt) ** self.gamma
        return (focal_weight * bce).mean()

## 7. Train / Evaluate Helpers

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_probs, all_labels = [], []
    criterion = nn.BCEWithLogitsLoss()
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        total_loss += loss.item() * len(y_batch)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(y_batch.cpu().numpy())
    return (total_loss / len(loader.dataset),
            np.array(all_probs),
            np.array(all_labels))


def best_threshold_for_recall(probs, labels, min_precision=0.3):
    """Find the lowest threshold that maximises recall subject to precision >= min_precision.
    Recall-first: we accept false alarms rather than miss a genuine alarm event.
    min_precision=0.2 prevents the degenerate solution of always predicting 1.
    """
    precisions, recalls, thresholds = precision_recall_curve(labels, probs)
    # thresholds has len N-1; precisions/recalls have len N
    valid = [(p, r, t) for p, r, t in zip(precisions[:-1], recalls[:-1], thresholds)
             if r >= min_recall]
    if not valid:
        return 0.4  # low fallback — biases toward predicting alarm
    best = max(valid, key=lambda x: x[1])  # maximise recall
    return best[2]


def pr_auc(probs, labels):
    return average_precision_score(labels, probs)

## 8. Manual Hyperparameter Search (on Train/Val Split)

In [ ]:
# (Old cleaning logic removed to save memory)

Cleaning sequences...
  Train  — min: -36.1420, max: 355.8733
  Val    — min: -36.0853,   max: 347.6487
  Test   — min: -36.1845,  max: 378.0753

Scaled train — mean: 0.0000, std: 1.0000
Data ready for training ✓


In [ ]:
# ── CHECKPOINT SAVE (run once after scaling succeeds) ─────────────────────────
import joblib, os

CKPT_DIR = '../checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

if isinstance(X_train_sc, LazySequenceArray):
    np.save(f'{CKPT_DIR}/X_train_sc_data.npy', X_train_sc.data_array)
    joblib.dump(X_train_sc.indices, f'{CKPT_DIR}/X_train_sc_indices.joblib')
else:
    np.save(f'{CKPT_DIR}/X_train_sc.npy', X_train_sc)

if isinstance(X_val_sc, LazySequenceArray):
    np.save(f'{CKPT_DIR}/X_val_sc_data.npy', X_val_sc.data_array)
    joblib.dump(X_val_sc.indices, f'{CKPT_DIR}/X_val_sc_indices.joblib')
else:
    np.save(f'{CKPT_DIR}/X_val_sc.npy', X_val_sc)

if isinstance(X_test_sc, LazySequenceArray):
    np.save(f'{CKPT_DIR}/X_test_sc_data.npy', X_test_sc.data_array)
    joblib.dump(X_test_sc.indices, f'{CKPT_DIR}/X_test_sc_indices.joblib')
else:
    np.save(f'{CKPT_DIR}/X_test_sc.npy', X_test_sc)

np.save(f'{CKPT_DIR}/y_train.npy', y_train)
np.save(f'{CKPT_DIR}/y_val.npy', y_val)
np.save(f'{CKPT_DIR}/y_test.npy', y_test)

if 'scaler_global' in globals():
    joblib.dump(scaler_global, f'{CKPT_DIR}/scaler.joblib')
elif 'scaler' in globals():
    joblib.dump(scaler, f'{CKPT_DIR}/scaler.joblib')
print("Feature scaling checkpoint saved!")


Checkpoint saved to ../checkpoints/
  X_train_sc : (942616, 20, 93)
  X_val_sc   : (201029, 20, 93)
  X_test_sc  : (185845, 20, 93)


In [ ]:
# ── Best Hyperparameters ─────────────────────────────────────
# Parameters are chosen to prevent overfitting while prioritising recall
# (catching every alarm is more important than avoiding false alarms).

best_params = {
    'hidden_units': 64,
    'num_layers': 2,
    'dropout': 0.3,
    'rec_dropout': 0.2,
    'lr': 1e-3,
    'batch_size': 256,
    'pos_weight': 50.0
}
# Note: L2 regularization is achieved via weight_decay=1e-4 in the Adam optimizer.
# pos_weight=3.0 means a false negative costs 3x a false positive in FocalLoss.


# ── Save best_params so subsequent cells work ────────────────────────────
import json
with open(f'{CKPT_DIR}/best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
print(f'best_params saved to {CKPT_DIR}/best_params.json')


Python(12131) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Running Optuna (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]

: 

## 9. 10-Fold GroupKFold Training with Best Hyperparameters

In [ ]:
gkf10 = GroupKFold(n_splits=N_FOLDS)

fold_results  = []
fold_histories = []  # loss curves per fold
best_models   = []   # saved state dicts

for fold_idx, (tr_idx, val_idx) in enumerate(
        gkf10.split(X_train_sc, y_train, g_train)):

    print(f'\n── Fold {fold_idx + 1}/{N_FOLDS} ──────────────────────')

    # Fixed MemoryError by using Subset instead of physical array slicing
    train_dataset = torch.utils.data.Subset(AlarmDataset(X_train_sc, y_train), tr_idx)
    val_dataset   = torch.utils.data.Subset(AlarmDataset(X_train_sc, y_train), val_idx)
    
    fold_labels = y_train[tr_idx]
    fold_class_counts = np.bincount(fold_labels.astype(int))
    fold_sample_weights = 1.0 / fold_class_counts[fold_labels.astype(int)]
    fold_sampler = torch.utils.data.WeightedRandomSampler(
        weights=fold_sample_weights,
        num_samples=len(fold_sample_weights),
        replacement=True
    )
    tr_loader = DataLoader(
        train_dataset,
        batch_size=best_params.get('batch_size', 256),
        sampler=fold_sampler
    )
    val_loader = DataLoader(val_dataset,
                            batch_size=best_params.get('batch_size', 32))

    model = make_model(best_params, INPUT_SIZE)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=best_params.get('lr', 1e-3))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=4, min_lr=1e-6)
    criterion = FocalLoss(
        gamma=2.0, pos_weight=best_params.get('pos_weight', POS_WEIGHT_DEFAULT))

    best_val_prauc = 0.0
    patience_ctr   = 0
    best_state     = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    history        = {'train_loss': [], 'val_loss': [], 'val_prauc': []}

    for epoch in range(EPOCHS):
        tr_loss = train_one_epoch(model, tr_loader, optimizer, criterion)
        val_loss, val_probs, val_labels = evaluate(model, val_loader)
        val_score = pr_auc(val_probs, val_labels)
        scheduler.step(val_score)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['val_prauc'].append(val_score)

        if val_score > best_val_prauc or epoch == 0:
            best_val_prauc = val_score
            best_state = {k: v.cpu().clone()
                         for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d} | '
                  f'Train Loss: {tr_loss:.4f} | '
                  f'Val Loss: {val_loss:.4f} | '
                  f'Val PR-AUC: {val_score:.4f}')

        if patience_ctr >= PATIENCE:
            print(f'  Early stop at epoch {epoch + 1}')
            break

    # Reload best weights and find best threshold on this val fold
    model.load_state_dict(best_state)
    _, val_probs_best, val_labels_best = evaluate(model, val_loader)
    thr = best_threshold_for_recall(val_probs_best, val_labels_best, min_precision=0.3)

    fold_results.append({
        'fold'        : fold_idx + 1,
        'val_pr_auc'  : best_val_prauc,
        'threshold'   : thr,
        'val_probs'   : val_probs_best,
        'val_labels'  : val_labels_best,
    })
    fold_histories.append(history)
    best_models.append(best_state)

    print(f'  Best Val PR-AUC: {best_val_prauc:.4f} | Threshold: {thr:.2f}')

print('\n── 10-Fold CV Complete ──────────────────────────────────')
mean_prauc = np.mean([r['val_pr_auc'] for r in fold_results])
std_prauc  = np.std([r['val_pr_auc']  for r in fold_results])
print(f'Mean Val PR-AUC: {mean_prauc:.4f} ± {std_prauc:.4f}')


── Fold 1/10 ──────────────────────


NameError: name 'best_params' is not defined

## 10. Loss Curves — All Folds

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()

for i, (hist, res) in enumerate(zip(fold_histories, fold_results)):
    ax = axes[i]
    epochs_ran = range(1, len(hist['train_loss']) + 1)
    ax.plot(epochs_ran, hist['train_loss'], label='Train Loss', color='steelblue')
    ax.plot(epochs_ran, hist['val_loss'],   label='Val Loss',   color='tomato')
    ax2 = ax.twinx()
    ax2.plot(epochs_ran, hist['val_prauc'], label='Val PR-AUC',
             color='forestgreen', linestyle='--', alpha=0.8)
    ax2.set_ylabel('PR-AUC', color='forestgreen', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='forestgreen')
    ax.set_title(f'Fold {res["fold"]}  (PR-AUC={res["val_pr_auc"]:.3f})', fontsize=9)
    ax.set_xlabel('Epoch', fontsize=8)
    ax.set_ylabel('Loss', fontsize=8)
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle('GRU Training — Loss & Val PR-AUC per Fold', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fold_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fold_loss_curves.png')

## 11. Per-Fold Metrics Summary

In [ ]:
summary_rows = []
for res in fold_results:
    thr   = res['threshold']
    preds = (res['val_probs'] >= thr).astype(int)
    labels = res['val_labels'].astype(int)
    report = classification_report(labels, preds, output_dict=True, zero_division=0)
    pos = report.get('1', report.get('1.0', {}))
    summary_rows.append({
        'Fold'      : res['fold'],
        'Threshold' : round(thr, 2),
        'PR-AUC'    : round(res['val_pr_auc'], 4),
        'Precision' : round(pos.get('precision', 0), 4),
        'Recall'    : round(pos.get('recall', 0), 4),
        'F1'        : round(pos.get('f1-score', 0), 4),
        'Accuracy'  : round(report['accuracy'], 4),
    })

summary_df = pd.DataFrame(summary_rows)
mean_row   = summary_df.drop(columns='Fold').mean().round(4).to_dict()
mean_row['Fold'] = 'MEAN'
std_row    = summary_df.drop(columns='Fold').std().round(4).to_dict()
std_row['Fold']  = 'STD'
summary_df = pd.concat([summary_df,
                         pd.DataFrame([mean_row, std_row])],
                        ignore_index=True)
print(summary_df.to_string(index=False))

## 12. Per-Fold Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()

for i, res in enumerate(fold_results):
    thr   = res['threshold']
    preds = (res['val_probs'] >= thr).astype(int)
    cm    = confusion_matrix(res['val_labels'].astype(int), preds)
    disp  = ConfusionMatrixDisplay(cm, display_labels=['No Alarm', 'Alarm'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(f'Fold {res["fold"]}  (thr={thr:.2f})', fontsize=9)
    axes[i].set_xlabel('Predicted', fontsize=8)
    axes[i].set_ylabel('Actual', fontsize=8)

plt.suptitle('Confusion Matrices — All 10 Folds (Val Set)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fold_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fold_confusion_matrices.png')

## 13. Precision-Recall Curves — All Folds

In [ ]:
plt.figure(figsize=(9, 7))
colors = plt.cm.tab10(np.linspace(0, 1, N_FOLDS))

for res, color in zip(fold_results, colors):
    prec, rec, _ = precision_recall_curve(res['val_labels'], res['val_probs'])
    plt.plot(rec, prec, color=color, alpha=0.7,
             label=f'Fold {res["fold"]} (AUC={res["val_pr_auc"]:.3f})')

plt.axhline(y=0.6, color='grey', linestyle=':', alpha=0.6, label='Recall=0.6 threshold')
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves — 10-Fold CV', fontsize=14)
plt.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: pr_curves.png')

## 14. Final Model — Train on Full Train Set, Evaluate on Validation & Test

In [ ]:
# Use median threshold from CV folds
final_threshold = float(np.median([r['threshold'] for r in fold_results]))
print(f'Final threshold (median of CV folds): {final_threshold:.2f}')

# Train final model
class_counts = np.bincount(y_train.astype(int))
sample_weights = 1.0 / class_counts[y_train.astype(int)]
sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)
full_tr_loader = DataLoader(
    AlarmDataset(X_train_sc, y_train),
    batch_size=best_params.get('batch_size', 256),
    sampler=sampler
)
val_loader_full = DataLoader(AlarmDataset(X_val_sc, y_val),
                              batch_size=best_params.get('batch_size', 32))
test_loader     = DataLoader(AlarmDataset(X_test_sc, y_test),
                              batch_size=best_params.get('batch_size', 32))

final_model = make_model(best_params, INPUT_SIZE)
optimizer   = torch.optim.Adam(
    final_model.parameters(), lr=best_params.get('lr', 1e-3))
scheduler   = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=4, min_lr=1e-6)
criterion   = FocalLoss(
    gamma=2.0, pos_weight=best_params.get('pos_weight', POS_WEIGHT_DEFAULT))

best_val_prauc  = 0.0
patience_ctr    = 0
final_best_state = {k: v.cpu().clone() for k, v in final_model.state_dict().items()}
final_history   = {'train_loss': [], 'val_loss': [], 'val_prauc': []}

print('Training final model...')
for epoch in range(EPOCHS):
    tr_loss = train_one_epoch(final_model, full_tr_loader, optimizer, criterion)
    val_loss, val_probs, val_labels = evaluate(final_model, val_loader_full)
    score = pr_auc(val_probs, val_labels)
    scheduler.step(score)

    final_history['train_loss'].append(tr_loss)
    final_history['val_loss'].append(val_loss)
    final_history['val_prauc'].append(score)

    if score > best_val_prauc or epoch == 0:
        best_val_prauc = score
        final_best_state = {k: v.cpu().clone()
                           for k, v in final_model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1

    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d} | '
              f'Train Loss: {tr_loss:.4f} | '
              f'Val Loss: {val_loss:.4f} | '
              f'Val PR-AUC: {score:.4f}')

    if patience_ctr >= PATIENCE:
        print(f'  Early stop at epoch {epoch + 1}')
        break

final_model.load_state_dict(final_best_state)
print(f'\nBest Val PR-AUC (final model): {best_val_prauc:.4f}')

## 15. Final Model — Loss Curve

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ep = range(1, len(final_history['train_loss']) + 1)
ax1.plot(ep, final_history['train_loss'], label='Train Loss', color='steelblue')
ax1.plot(ep, final_history['val_loss'],   label='Val Loss',   color='tomato')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(ep, final_history['val_prauc'], label='Val PR-AUC',
         color='forestgreen', linestyle='--')
ax2.set_ylabel('PR-AUC', color='forestgreen')
ax2.tick_params(axis='y', labelcolor='forestgreen')
ax2.legend(loc='upper right')

plt.title('Final Model — Training Curve')
plt.tight_layout()
plt.savefig('final_model_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: final_model_loss_curve.png')

## 16. Validation Set — Full Evaluation

In [ ]:
_, val_probs_final, val_labels_final = evaluate(final_model, val_loader_full)

# Fine-tune threshold on actual validation set
val_threshold = best_threshold_for_recall(
    val_probs_final, val_labels_final, min_precision=0.3)
print(f'Validation-tuned threshold: {val_threshold:.2f}')

val_preds = (val_probs_final >= val_threshold).astype(int)

print('\n── Validation Classification Report ─────────────────────')
print(classification_report(val_labels_final.astype(int), val_preds,
                             target_names=['No Alarm', 'Alarm']))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm_val = confusion_matrix(val_labels_final.astype(int), val_preds)
ConfusionMatrixDisplay(cm_val, display_labels=['No Alarm', 'Alarm']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Validation Confusion Matrix (thr={val_threshold:.2f})')

# PR Curve
prec_v, rec_v, _ = precision_recall_curve(val_labels_final, val_probs_final)
prauc_v = auc(rec_v, prec_v)
axes[1].plot(rec_v, prec_v, color='steelblue', lw=2)
axes[1].fill_between(rec_v, prec_v, alpha=0.15, color='steelblue')
axes[1].set_title(f'Validation PR Curve (AUC={prauc_v:.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('validation_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: validation_evaluation.png')

## 17. Test Set — Final Evaluation

In [ ]:
_, test_probs, test_labels = evaluate(final_model, test_loader)
test_preds = (test_probs >= val_threshold).astype(int)

print('\n── TEST SET Classification Report ───────────────────────')
print(classification_report(test_labels.astype(int), test_preds,
                             target_names=['No Alarm', 'Alarm']))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm_test = confusion_matrix(test_labels.astype(int), test_preds)
ConfusionMatrixDisplay(cm_test, display_labels=['No Alarm', 'Alarm']).plot(
    ax=axes[0], colorbar=False, cmap='Oranges')
axes[0].set_title(f'Test Confusion Matrix (thr={val_threshold:.2f})')

# PR Curve
prec_t, rec_t, _ = precision_recall_curve(test_labels, test_probs)
prauc_t = auc(rec_t, prec_t)
axes[1].plot(rec_t, prec_t, color='darkorange', lw=2)
axes[1].fill_between(rec_t, prec_t, alpha=0.15, color='darkorange')
axes[1].set_title(f'Test PR Curve (AUC={prauc_t:.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('test_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: test_evaluation.png')
print(f"\nExplicit Test Metrics:")
print(f"PR-AUC: {prauc_t:.4f}")

## 17b. Overfitting Diagnostics

In [ ]:
_, train_probs_final, train_labels_final = evaluate(final_model, full_tr_loader)
prec_train, rec_train, _ = precision_recall_curve(train_labels_final, train_probs_final)
prauc_train = auc(rec_train, prec_train)

gap = prauc_train - prauc_v
print("── Overfitting Diagnostics ──")
print(f"Train PR-AUC: {prauc_train:.4f}")
print(f"Validation PR-AUC: {prauc_v:.4f}")
print(f"Gap: {gap:.4f}")

if gap > 0.02:
    print("Potential overfitting detected")
else:
    print("Generalization appears acceptable")
    
# Free memory
del train_probs_final, train_labels_final
import gc; gc.collect()

## 17c. Multiple Threshold Evaluation

In [ ]:
print("\n── Evaluating Multiple Thresholds on Test Set ──")
thresholds_to_test = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
multi_thr_results = []
for t in thresholds_to_test:
    t_preds = (test_probs >= t).astype(int)
    report = classification_report(test_labels.astype(int), t_preds, output_dict=True, zero_division=0)
    pos = report.get('1', report.get('1.0', {}))
    multi_thr_results.append({
        'Threshold': t,
        'Precision': round(pos.get('precision', 0), 4),
        'Recall': round(pos.get('recall', 0), 4),
        'F1-Score': round(pos.get('f1-score', 0), 4),
    })

import pandas as pd
multi_thr_df = pd.DataFrame(multi_thr_results)
print(multi_thr_df.to_string(index=False))

## 18. Save Model & Scaler

In [ ]:
import joblib

torch.save({
    'model_state_dict': final_best_state,
    'best_params'     : best_params,
    'threshold'       : val_threshold,
    'input_size'      : INPUT_SIZE,
    'lookback'        : LOOKBACK,
    'feature_cols'    : FEATURE_COLS,
}, 'gru_alarm_model.pt')

joblib.dump(scaler_global, 'scaler.pkl')

print('Saved: gru_alarm_model.pt')
print('Saved: scaler.pkl')
print(f'\nFinal threshold applied at inference: {val_threshold:.2f}')
print(f'Best hyperparameters: {best_params}')

In [ ]:
# ── 19. Inference Pipeline (Testing on 13 Columns) ──────────────────────────
# Requirement: "testing should be on only 13 columns"
# This function demonstrates how to take a raw 13-column DataFrame, compute the
# required 93 features dynamically, scale them, and produce predictions.

def run_inference_pipeline(raw_data_13_cols, selected_features, scaler, model, lookback, device):
    """
    Args:
        raw_data_13_cols: DataFrame containing the 13 base sensor columns + timestamp
        selected_features: List of 93 feature names the model was trained on
        scaler: The fitted StandardScaler from training
        model: The trained GRU model
        lookback: The sequence lookback length (e.g., 20)
    Returns:
        DataFrame with predictions
    """
    # 1. Feature Engineering (Dynamically compute only what's needed)
    base_cols = [c for c in raw_data_13_cols.columns if c != 'timestamp']
    df_eng = create_engineered_features(raw_data_13_cols, base_cols, selected_features)
    
    # Ensure all selected features exist (fill missing with 0 or mean if any edge cases)
    for f in selected_features:
        if f not in df_eng.columns:
            df_eng[f] = 0.0
            
    # Extract feature matrix
    X_features = df_eng[selected_features].values.astype(np.float32)
    
    # Replace infs and nans (similar to clean_sequences)
    X_features[~np.isfinite(X_features)] = np.nan
    col_medians = np.nanmedian(X_features, axis=0)
    for i in range(X_features.shape[1]):
        mask = ~np.isfinite(X_features[:, i])
        if mask.any():
            X_features[mask, i] = col_medians[i] if not np.isnan(col_medians[i]) else 0.0
            
    # Scale features
    X_scaled = scaler.transform(X_features)
    
    # Build sequences (assuming stride=1 for testing)
    sequences = []
    timestamps = []
    
    for i in range(len(X_scaled) - lookback + 1):
        sequences.append(X_scaled[i:i + lookback])
        if 'timestamp' in raw_data_13_cols.columns:
            timestamps.append(raw_data_13_cols['timestamp'].iloc[i + lookback - 1])
        
    if not sequences:
        print("Not enough data to form a single sequence.")
        return pd.DataFrame()
        
    X_tensor = torch.tensor(np.array(sequences), dtype=torch.float32).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        probs = torch.sigmoid(logits).cpu().numpy()
        
    # Format output
    out_df = pd.DataFrame({'alarm_probability': probs})
    if timestamps:
        out_df['timestamp'] = timestamps
        out_df = out_df[['timestamp', 'alarm_probability']]
        
    # Apply final threshold (global variable from notebook)
    try:
        out_df['alarm_prediction'] = (probs >= val_threshold).astype(int)
    except NameError:
        out_df['alarm_prediction'] = (probs >= 0.6).astype(int) # fallback
        
    return out_df

# Example Usage (Uncomment to test):
# raw_test_data = df[BASE_COLS + ['timestamp']].tail(100).copy() # Simulating a 13-col raw input
# predictions = run_inference_pipeline(raw_test_data, SELECTED_FEATURES, scaler_global, final_model, LOOKBACK, DEVICE)
# print(predictions.head())


In [ ]:
# ── 19. Inference Pipeline (Testing on 13 Columns) ──────────────────────────
# Requirement: "testing should be on only 13 columns"
# This function demonstrates how to take a raw 13-column DataFrame, compute the
# required 93 features dynamically, scale them, and produce predictions.

def run_inference_pipeline(raw_data_13_cols, selected_features, scaler, model, lookback, device):
    """
    Args:
        raw_data_13_cols: DataFrame containing the 13 base sensor columns + timestamp
        selected_features: List of 93 feature names the model was trained on
        scaler: The fitted StandardScaler from training
        model: The trained GRU model
        lookback: The sequence lookback length (e.g., 20)
    Returns:
        DataFrame with predictions
    """
    # 1. Feature Engineering (Dynamically compute only what's needed)
    base_cols = [c for c in raw_data_13_cols.columns if c != 'timestamp']
    df_eng = create_engineered_features(raw_data_13_cols, base_cols, selected_features)
    
    # Ensure all selected features exist (fill missing with 0 or mean if any edge cases)
    for f in selected_features:
        if f not in df_eng.columns:
            df_eng[f] = 0.0
            
    # Extract feature matrix
    X_features = df_eng[selected_features].values.astype(np.float32)
    
    # Replace infs and nans (similar to clean_sequences)
    X_features[~np.isfinite(X_features)] = np.nan
    col_medians = np.nanmedian(X_features, axis=0)
    for i in range(X_features.shape[1]):
        mask = ~np.isfinite(X_features[:, i])
        if mask.any():
            X_features[mask, i] = col_medians[i] if not np.isnan(col_medians[i]) else 0.0
            
    # Scale features
    X_scaled = scaler.transform(X_features)
    
    # Build sequences (assuming stride=1 for testing)
    sequences = []
    timestamps = []
    
    for i in range(len(X_scaled) - lookback + 1):
        sequences.append(X_scaled[i:i + lookback])
        if 'timestamp' in raw_data_13_cols.columns:
            timestamps.append(raw_data_13_cols['timestamp'].iloc[i + lookback - 1])
        
    if not sequences:
        print("Not enough data to form a single sequence.")
        return pd.DataFrame()
        
    X_tensor = torch.tensor(np.array(sequences), dtype=torch.float32).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        probs = torch.sigmoid(logits).cpu().numpy()
        
    # Format output
    out_df = pd.DataFrame({'alarm_probability': probs})
    if timestamps:
        out_df['timestamp'] = timestamps
        out_df = out_df[['timestamp', 'alarm_probability']]
        
    # Apply final threshold (global variable from notebook)
    try:
        out_df['alarm_prediction'] = (probs >= val_threshold).astype(int)
    except NameError:
        out_df['alarm_prediction'] = (probs >= 0.6).astype(int) # fallback
        
    return out_df

# Example Usage (Uncomment to test):
# raw_test_data = df[BASE_COLS + ['timestamp']].tail(100).copy() # Simulating a 13-col raw input
# predictions = run_inference_pipeline(raw_test_data, SELECTED_FEATURES, scaler_global, final_model, LOOKBACK, DEVICE)
# print(predictions.head())
